In [1]:
import pandas as pd
print(pd.__version__)

3.0.6


In [5]:
df_03 = pd.read_csv('/Users/admin/match-engine/data/raw/E0_0304.csv')
df_03.shape

ParserError: Error tokenizing data. C error: Expected 57 fields in line 305, saw 72


In [7]:
from os import path


path = "../data/raw/E0_0304.csv"

with open(path) as f:
    lines = f.readlines()

print(len(lines[0].split(",")),lines[0])
print(len(lines[304].split(",")), lines[304])    

57 Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A,GBH,GBD,GBA,IWH,IWD,IWA,LBH,LBD,LBA,SOH,SOD,SOA,SBH,SBD,SBA,WHH,WHD,WHA,GB>2.5,GB<2.5,B365>2.5,B365<2.5,GBAHH,GBAHA,GBAH,LBAHH,LBAHA,LBAH,B365AHH,B365AHA,B365AH

72 E0,03/04/04,Tottenham,Chelsea,0,1,A,0,1,A,S Bennett,12,10,7,7,10,10,5,4,2,0,0,0,4,3.25,1.909,3.5,3.25,2.05,3.5,3.1,1.9,3,3.2,2.1,3.75,3.4,1.91,3.6,3.2,1.9,3.75,3.3,1.8,1.79,1.9,1.9,1.9,2.05,1.7,0.25,1.8,2.05,0.5,1.85,2.05,0.5,,,,,,,,,,,,,,,



In [10]:
header_len = len(lines[0].split(","))
bad_rows = 0
extras_with_data = 0

for line in lines[1:]:
    values = line.strip().split(",")
    if len(values) != header_len:
        bad_rows += 1
        extras = values[header_len:]
        if any(v != "" for v in extras):
            extras_with_data += 1

print("Rows with wrong length:", bad_rows)
print("Of those, rows with real data in extras:", extras_with_data)            

Rows with wrong length: 45
Of those, rows with real data in extras: 0


**Finding 1:** 45 rows in 2003/04 have 15 extra empty values (trailing commas).
Checked: none contain data. **Fix:** cut every row to the header length.

In [13]:
df_03 = pd.read_csv(
    path,
    engine="python",
    on_bad_lines=lambda row: row[:header_len]
)
df_03.shape


(380, 57)

**Check:** 380 rows (20 teams × 19 home games) ✓

In [14]:
df_03.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,B365<2.5,GBAHH,GBAHA,GBAH,LBAHH,LBAHA,LBAH,B365AHH,B365AHA,B365AH
0,E0,16/08/03,Arsenal,Everton,2,1,H,1,0,H,...,2.10,1.90,1.90,-1.25,1.90,1.95,-1.25,1.975,1.925,-1.25
1,E0,16/08/03,Birmingham,Tottenham,1,0,H,1,0,H,...,1.85,2.05,1.75,-0.25,1.75,2.10,0.00,2.100,1.800,-0.25
2,E0,16/08/03,Blackburn,Wolves,5,1,H,2,0,H,...,1.90,1.95,1.85,-0.75,2.00,1.85,-0.75,1.950,1.950,-0.75
3,E0,16/08/03,Fulham,Middlesbrough,3,2,H,1,1,D,...,1.70,1.80,2.00,0.00,1.85,2.00,0.00,2.100,1.800,-0.25
4,E0,16/08/03,Leicester,Southampton,2,2,D,2,0,H,...,1.80,1.85,1.95,0.00,1.85,2.00,0.00,1.850,2.050,0.00


**Finding 2:** Dates are DD/MM/YY (two-digit year). Risk: misread as
month-first, or wrong century. **Fix:** parse explicitly with format
"%d/%m/%y". Check other seasons use the same format.


In [15]:
df_03.tail()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,B365<2.5,GBAHH,GBAHA,GBAH,LBAHH,LBAHA,LBAH,B365AHH,B365AHA,B365AH
375,E0,15/05/04,Chelsea,Leeds,1,0,H,1,0,H,...,2.05,1.30,2.96,-0.50,1.70,2.14,-1.50,NaN,NaN,NaN
376,E0,15/05/04,Liverpool,Newcastle,1,1,D,0,1,A,...,1.80,2.00,1.65,-0.50,NaN,NaN,NaN,2.100,1.800,-0.50
377,E0,15/05/04,Man City,Everton,5,1,H,3,0,H,...,1.90,1.95,1.69,-0.50,1.72,2.12,-0.25,1.875,2.025,-0.25
378,E0,15/05/04,Portsmouth,Middlesbrough,5,1,H,3,1,H,...,1.80,1.95,1.90,-0.25,2.00,1.84,-0.25,2.075,1.825,-0.25
379,E0,15/05/04,Wolves,Tottenham,0,2,A,0,1,A,...,2.00,1.75,2.10,0.00,NaN,NaN,NaN,NaN,NaN,NaN


**Finding 3:** Some odds columns have missing values (NaN), varying by
bookmaker. Need to check which bookmakers have complete data before
choosing one for backtesting.

In [16]:
df_03.isna().sum()

Div          0
Date         0
HomeTeam     0
AwayTeam     0
FTHG         0
FTAG         0
FTR          0
HTHG         0
HTAG         0
HTR          0
Referee      0
HS           0
AS           0
HST          0
AST          0
HF           0
AF           0
HC           0
AC           0
HY           0
AY           0
HR           0
AR           0
B365H        0
B365D        0
B365A        0
GBH          0
GBD          0
GBA          0
IWH          0
IWD          0
IWA          0
LBH          1
LBD          1
LBA          1
SOH          0
SOD          0
SOA          0
SBH          0
SBD          0
SBA          0
WHH          0
WHD          0
WHA          0
GB>2.5       1
GB<2.5       1
B365>2.5     2
B365<2.5     2
GBAHH        0
GBAHA        9
GBAH         9
LBAHH       28
LBAHA       28
LBAH        28
B365AHH     20
B365AHA     20
B365AH      20
dtype: int64

In [17]:
missing = df_03.isna().sum()
missing[missing > 0]

LBH          1
LBD          1
LBA          1
GB>2.5       1
GB<2.5       1
B365>2.5     2
B365<2.5     2
GBAHA        9
GBAH         9
LBAHH       28
LBAHA       28
LBAH        28
B365AHH     20
B365AHA     20
B365AH      20
dtype: int64

**Finding 4:** Core columns (Date, teams, goals, result) have no gaps.
Home/draw/away odds complete for all bookmakers except Ladbrokes (1 gap).
**Candidate for backtesting:** Bet365 (B365H/D/A). Must confirm it exists
in later seasons.

In [21]:
df_12 = pd.read_csv("../data/raw/E0_1213.csv")
df_12.shape

(380, 74)

In [32]:
df_12.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
0,E0,18/08/12,Arsenal,Sunderland,0,0,D,0,0,D,...,2.05,23,-1.25,2.02,1.96,1.96,1.91,1.44,4.72,8.71
1,E0,18/08/12,Fulham,Norwich,5,0,H,2,0,H,...,1.92,21,-0.50,1.83,1.80,2.14,2.09,1.84,3.75,4.75
2,E0,18/08/12,Newcastle,Tottenham,2,1,H,0,0,D,...,1.92,23,0.00,1.93,1.88,2.03,1.97,2.83,3.35,2.72
3,E0,18/08/12,QPR,Swansea,0,5,A,0,1,A,...,1.73,22,-0.25,1.80,1.75,2.21,2.13,2.00,3.53,4.15
4,E0,18/08/12,Reading,Stoke,1,1,D,0,1,A,...,1.67,24,-0.25,2.07,2.01,1.91,1.86,2.47,3.30,3.22


**Finding 5:** 2012/13 has 74 columns vs 57 in 2003/04. Loaded without
the trailing-comma problem.

In [22]:
cols_03 = set(df_03.columns)
cols_12 = set(df_12.columns)

print("Only in 2003/04:", sorted(cols_03 - cols_12))
print("Only in 2012/13:", sorted(cols_12 - cols_03))

Only in 2003/04: ['B365<2.5', 'B365>2.5', 'B365AH', 'B365AHA', 'B365AHH', 'GB<2.5', 'GB>2.5', 'GBAH', 'GBAHA', 'GBAHH', 'LBAH', 'LBAHA', 'LBAHH', 'SBA', 'SBD', 'SBH', 'SOA', 'SOD', 'SOH']
Only in 2012/13: ['BSA', 'BSD', 'BSH', 'BWA', 'BWD', 'BWH', 'Bb1X2', 'BbAH', 'BbAHh', 'BbAv<2.5', 'BbAv>2.5', 'BbAvA', 'BbAvAHA', 'BbAvAHH', 'BbAvD', 'BbAvH', 'BbMx<2.5', 'BbMx>2.5', 'BbMxA', 'BbMxAHA', 'BbMxAHH', 'BbMxD', 'BbMxH', 'BbOU', 'PSA', 'PSCA', 'PSCD', 'PSCH', 'PSD', 'PSH', 'SJA', 'SJD', 'SJH', 'VCA', 'VCD', 'VCH']


In [ ]:
cols_03 = set(df_03.columns)
cols_12 = set(df_12.columns)

print("only in 2003/04:", sorted(cols_03 - cols_12))
print("only in 2012/13:", sorted(cols_12 - cols_03))


only in 2003/04: ['B365<2.5', 'B365>2.5', 'B365AH', 'B365AHA', 'B365AHH', 'GB<2.5', 'GB>2.5', 'GBAH', 'GBAHA', 'GBAHH', 'LBAH', 'LBAHA', 'LBAHH', 'SBA', 'SBD', 'SBH', 'SOA', 'SOD', 'SOH']
only in 2012/13: ['BSA', 'BSD', 'BSH', 'BWA', 'BWD', 'BWH', 'Bb1X2', 'BbAH', 'BbAHh', 'BbAv<2.5', 'BbAv>2.5', 'BbAvA', 'BbAvAHA', 'BbAvAHH', 'BbAvD', 'BbAvH', 'BbMx<2.5', 'BbMx>2.5', 'BbMxA', 'BbMxAHA', 'BbMxAHH', 'BbMxD', 'BbMxH', 'BbOU', 'PSA', 'PSCA', 'PSCD', 'PSCH', 'PSD', 'PSH', 'SJA', 'SJD', 'SJH', 'VCA', 'VCD', 'VCH']


**Finding 6:** Column set changes between seasons. Bookmakers come and go.
Bet365 1X2 odds (B365H/D/A) present in 2003/04 and 2012/13.
**Fix:** cleaner must keep only a fixed list of needed columns, not rely
on column positions or counts.

In [28]:
df_25 = pd.read_csv("../data/raw/E0_2526.csv")
df_25.shape

(380, 132)

In [30]:
df_25.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


**Finding 2 (updated):** Date format varies. 2003/04 uses DD/MM/YY,
2025/26 uses DD/MM/YYYY. **Fix:** try both formats; fail loudly if
neither works.

**Finding 8:** 2025/26 has a Time column (kick-off). Not present in
2003/04. Useful for consistent ordering of same-day matches.

**Finding 9:** 2012/13 uses DD/MM/YY and has no Time column. Format
change happened between 2012/13 and 2025/26; exact season unknown.
Cleaner must not assume format by season.

In [31]:
df_25.tail()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
375,E0,24/05/2026,16:00,Man City,Aston Villa,1,2,A,1,0,...,1.83,2.03,NaN,NaN,1.83,2.05,1.78,1.98,1.90,2.09
376,E0,24/05/2026,16:00,Nott'm Forest,Bournemouth,1,1,D,1,0,...,1.90,1.95,NaN,NaN,1.90,1.95,1.81,1.91,1.94,2.04
377,E0,24/05/2026,16:00,Sunderland,Chelsea,2,1,H,1,0,...,1.80,2.05,NaN,NaN,1.80,2.05,1.75,1.99,1.87,2.12
378,E0,24/05/2026,16:00,Tottenham,Everton,1,0,H,1,0,...,1.93,1.93,NaN,NaN,1.93,1.93,1.85,1.87,1.96,2.03
379,E0,24/05/2026,16:00,West Ham,Leeds,3,0,H,0,0,...,1.93,1.93,NaN,NaN,1.93,1.93,1.86,1.89,1.97,2.01


In [33]:
missing = df_25.isna().sum()
missing[missing > 0]

BFDH          1
BFDD          1
BFDA          1
BMGMH         2
BMGMD         2
BMGMA         2
BVH           2
BVD           2
BVA           2
CLH          98
CLD          98
CLA          98
LBH          94
LBD          94
LBA          94
PSH         170
PSD         170
PSA         170
BFEH         20
BFED         20
BFEA         20
P>2.5       170
P<2.5       170
BFE>2.5      20
BFE<2.5      20
AHh           1
PAHH        170
PAHA        170
BFEAHH       20
BFEAHA       20
BFDCH         8
BFDCD         8
BFDCA         8
BVCH          8
BVCD          8
BVCA          8
CLCH        120
CLCD        120
CLCA        120
LBCH         99
LBCD         99
LBCA         99
PSCH        170
PSCD        170
PSCA        170
BFECH        22
BFECD        22
BFECA        22
PC>2.5      170
PC<2.5      170
BFEC>2.5     22
BFEC<2.5     22
PCAHH       170
PCAHA       170
BFECAHH      22
BFECAHA      22
dtype: int64

**Finding 10:** 2025/26 core columns complete. Pinnacle odds missing for
170/380 matches. Several other bookmakers have gaps.
**Decision:** use Bet365 (B365H/D/A) for backtesting, pending confirmation
it exists in 2025/26.

In [34]:
all(col in df_25.columns for col in ["B365H", "B365D", "B365A"])

True

In [29]:
cols_03 = set(df_03.columns)
cols_12 = set(df_12.columns)
cols_25 = set(df_25.columns)

print("only in 2003/04"), sorted(cols_03 - cols_12 - cols_25)
print("only in 2012/13"), sorted(cols_12 - cols_03 - cols_25)
print("only in 2025/26"), sorted(cols_25 - cols_12 - cols_03)

only in 2003/04
only in 2012/13
only in 2025/26


(None,
 ['AHCh',
  'AHh',
  'Avg<2.5',
  'Avg>2.5',
  'AvgA',
  'AvgAHA',
  'AvgAHH',
  'AvgC<2.5',
  'AvgC>2.5',
  'AvgCA',
  'AvgCAHA',
  'AvgCAHH',
  'AvgCD',
  'AvgCH',
  'AvgD',
  'AvgH',
  'B365C<2.5',
  'B365C>2.5',
  'B365CA',
  'B365CAHA',
  'B365CAHH',
  'B365CD',
  'B365CH',
  'BFDA',
  'BFDCA',
  'BFDCD',
  'BFDCH',
  'BFDD',
  'BFDH',
  'BFE<2.5',
  'BFE>2.5',
  'BFEA',
  'BFEAHA',
  'BFEAHH',
  'BFEC<2.5',
  'BFEC>2.5',
  'BFECA',
  'BFECAHA',
  'BFECAHH',
  'BFECD',
  'BFECH',
  'BFED',
  'BFEH',
  'BMGMA',
  'BMGMCA',
  'BMGMCD',
  'BMGMCH',
  'BMGMD',
  'BMGMH',
  'BVA',
  'BVCA',
  'BVCD',
  'BVCH',
  'BVD',
  'BVH',
  'BWCA',
  'BWCD',
  'BWCH',
  'CLA',
  'CLCA',
  'CLCD',
  'CLCH',
  'CLD',
  'CLH',
  'LBCA',
  'LBCD',
  'LBCH',
  'Max<2.5',
  'Max>2.5',
  'MaxA',
  'MaxAHA',
  'MaxAHH',
  'MaxC<2.5',
  'MaxC>2.5',
  'MaxCA',
  'MaxCAHA',
  'MaxCAHH',
  'MaxCD',
  'MaxCH',
  'MaxD',
  'MaxH',
  'P<2.5',
  'P>2.5',
  'PAHA',
  'PAHH',
  'PC<2.5',
  'PC>2.5',
  'PCAH

In [35]:
teams_03 = set(df_03["HomeTeam"])
teams_12 = set(df_12["HomeTeam"])
teams_25 = set(df_25["HomeTeam"])

print(sorted(teams_03))
print(sorted(teams_12))
print(sorted(teams_25))

['Arsenal', 'Aston Villa', 'Birmingham', 'Blackburn', 'Bolton', 'Charlton', 'Chelsea', 'Everton', 'Fulham', 'Leeds', 'Leicester', 'Liverpool', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Portsmouth', 'Southampton', 'Tottenham', 'Wolves']
['Arsenal', 'Aston Villa', 'Chelsea', 'Everton', 'Fulham', 'Liverpool', 'Man City', 'Man United', 'Newcastle', 'Norwich', 'QPR', 'Reading', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'West Brom', 'West Ham', 'Wigan']
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Leeds', 'Liverpool', 'Man City', 'Man United', 'Newcastle', "Nott'm Forest", 'Sunderland', 'Tottenham', 'West Ham', 'Wolves']


#	Finding	Fix
1	45 rows in 2003/04 have 15 extra empty values (trailing commas). None contain data.	Cut every row to the header length.
2	Date format varies: DD/MM/YY in older seasons, DD/MM/YYYY in 2025/26.	Try both formats; fail loudly if neither works.
3	Some odds columns have missing values, varying by bookmaker.	Check completeness before choosing a bookmaker.
4	2003/04 core columns (Date, teams, goals, result) have no gaps. Bet365 complete.	None needed. Bet365 is the candidate.
5	2012/13 has 74 columns vs 57. No trailing-comma problem.	Finding 1 fix must only apply where needed.
6	Bookmakers come and go between seasons.	Select columns by name, never by position or count.
7	2025/26 has 132 columns.	Covered by Finding 6.
8	2025/26 has a Time column; 2003/04 doesn't.	Decide in spec how to order same-day matches.
9	2012/13 uses DD/MM/YY with no Time column. Format changed some time after.	Don't assume format by season.
10	Pinnacle odds missing for 170/380 matches in 2025/26.	Use Bet365 (B365H/D/A), confirmed in all three seasons.
11	Team names consistent across the 3 sampled seasons, but not proven for all.	Strip whitespace; review all unique names after combining.